In [ ]:
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)


project root: /home/temari/god please no diploma/restore_punct


## Run config

In [ ]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

# ── v2: 4 epochs + 10% warmup (val loss was still dropping at epoch 3)
cfg = RunConfig(
    name          = "03_crf_transitions_mined_v2",
    architecture  = "bert+crf",
    loss          = "crf",
    train_files   = [f"{DATA_DIR}/train_all_rare_punct.json"],
    val_files     = [f"{DATA_DIR}/val_internal.json"],
    num_epochs    = 4,
    learning_rate = 2e-5,
    warmup_ratio  = 0.1,
    crf_init_transitions = True,
    crf_aux_loss         = "none",
    tags          = {"stage": "3-improved-base"},
)
print(cfg)


RunConfig(name='03_crf_transitions_mined_v2', architecture='bert+crf', loss='crf', train_files=['/home/temari/god please no diploma/restore_punct/data/train_all_rare_punct.json'], val_files=['/home/temari/god please no diploma/restore_punct/data/val_internal.json'], replay_files=[], init_from=None, num_epochs=4, learning_rate=2e-05, benchmarks=None, seed=42, crf_init_transitions=True, crf_aux_loss='none', crf_aux_weight=0.2, crf_aux_gamma=2.0, o_mask_prob=0.0, warmup_ratio=0.1, tags={'stage': '3-improved-base'})


## Train

In [3]:
from pipeline.training import train_run
model = train_run(cfg)


[03_crf_transitions_mined_v2] architecture=bert+crf loss=crf epochs=4 lr=2e-05


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[03_crf_transitions_mined_v2] computing empirical CRF transitions from ['/home/temari/god please no diploma/restore_punct/data/train_all_rare_punct.json']...
[03_crf_transitions_mined_v2] CRF transitions warm-started from data


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/home/temari/anaconda3/envs/neural/lib/python3.13/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:614.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,41.760869,22.537052,0.961541,0.962901,0.961806,0.962901
2,36.569336,20.611662,0.964682,0.965907,0.964865,0.965907
3,32.870791,20.118853,0.965796,0.966863,0.966013,0.966863
4,30.925952,20.317081,0.966234,0.967220,0.966472,0.967220


Model saved -> /home/temari/god please no diploma/restore_punct/models/03_crf_transitions_mined_v2


## Benchmark + save results

In [4]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    print(f"{test_name:>14s} | F1={wa.get('f1-score', 0):.4f} P={wa.get('precision', 0):.4f} R={wa.get('recall', 0):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[03_crf_transitions_mined_v2] evaluating on General_Test (n=569)


Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

[03_crf_transitions_mined_v2] evaluating on GERA_Test (n=1259)


Evaluating:   0%|          | 0/315 [00:00<?, ?it/s]

[03_crf_transitions_mined_v2] evaluating on LORuGEC_Test (n=603)


Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]

Updated /home/temari/god please no diploma/restore_punct/results/models_db.json (entry: 03_crf_transitions_mined_v2)
Wrote /home/temari/god please no diploma/restore_punct/results/03_crf_transitions_mined_v2.json
Wrote /home/temari/god please no diploma/restore_punct/results/03_crf_transitions_mined_v2.xlsx
  General_Test | F1=0.9533 P=0.9523 R=0.9552
     GERA_Test | F1=0.9588 P=0.9594 R=0.9597
  LORuGEC_Test | F1=0.9712 P=0.9717 R=0.9715


## Example run

In [5]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела".

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань уже".

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.



## Stats

In [6]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()

Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 15
  Yandex runs : 1


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'